# Basic ifcfill usage

This notebook shows how to use `IFCTransformer` with a small pandas DataFrame. It covers type inference, missing-value imputation, datetime conversion, constant column removal, manual type overrides, and inverse transformation.

## Setup

Install ifcfill before running this notebook:

```bash
pip install ifcfill
```

When working from a local clone, install it in editable mode with the optional example dependencies:

```bash
pip install -e ".[examples]"
```

In [ ]:
import pandas as pd

from ifcfill import IFCTransformer

## Create sample data

The sample table includes integer-like values, floats, categoricals, datetimes, missing values, and one constant column.

In [ ]:
df = pd.DataFrame(
    {
        "age": [25, 30, None, 40, 35],
        "salary": [50_000.50, None, 75_000.00, 90_000.25, 62_000.00],
        "city": ["London", None, "Paris", "London", "Amman"],
        "joined": pd.to_datetime(
            ["2020-01-01", "2021-06-15", None, "2023-03-10", "2022-11-01"]
        ),
        "zip_code": ["00123", "00456", None, "00123", "07890"],
        "active": ["yes", "yes", "yes", "yes", "yes"],
    }
)

df

## Fit and transform

`IFCTransformer` learns column types and fill values during `fit`, then returns a transformed DataFrame with no missing values in the inferred IFC columns. This example enables categorical label encoding so categorical variables become integer codes that can be decoded by `inverse_transform`.

In [ ]:
transformer = IFCTransformer(
    col_types={"zip_code": "categorical"},
    int_fill="median",
    float_fill="mean",
    cat_fill="constant",
    cat_constant="UNKNOWN",
    cat_encoding="label",
    datetime_anchor="1970-01-01",
    datetime_unit="D",
)

transformed = transformer.fit_transform(df)
transformed

The constant `active` column is dropped from the transformed output. The `joined` datetime column is represented as integer days from the configured anchor date. Categorical columns such as `city` and `zip_code` are represented as integer label codes.

## Inspect what ifcfill learned

In [ ]:
transformer.column_types_

In [ ]:
transformer.fill_values_

In [ ]:
transformer.dropped_constants_

In [ ]:
transformer.category_mappings_

In [ ]:
transformer.missing_report_

## Restore the original structure

`inverse_transform` restores dropped constant columns and the original column order. With `restore_missing=True`, it also reintroduces missing values at the same rates observed during `fit`.

In [ ]:
restored = transformer.inverse_transform(
    transformed,
    restore_missing=True,
    random_state=42,
)

restored

## Use the same fitted transformer on new data

After fitting once, call `transform` on another DataFrame with the same columns to apply the learned rules.

In [ ]:
new_df = pd.DataFrame(
    {
        "age": [28, None],
        "salary": [70_000.00, None],
        "city": [None, "Paris"],
        "joined": pd.to_datetime(["2024-02-01", None]),
        "zip_code": [None, "00123"],
        "active": ["yes", "yes"],
    }
)

transformer.transform(new_df)